# **Project 05: Ensemble Machine Learning – Wine Dataset**

**Name:** Sabriya Sowers  
**Date:** November 15, 2025  

## **Introduction**
This analysis applies ensemble machine learning techniques to the Wine dataset to examine how model aggregation can enhance predictive performance compared to individual learners. The goal is to build, evaluate, and compare multiple ensemble approaches to understand how well they capture meaningful patterns within the dataset’s chemical attributes. By leveraging the combined strengths of multiple models, ensemble methods offer improved stability, reduced variance, and more dependable predictive accuracy—making them highly effective for practical, real-world analytical workflows.

## Section 1. Import and Inspect the Data

In [23]:
# Work with tabular data (rows and columns)
import pandas as pd

# Helpful tools for numbers, arrays, and math
import numpy as np

# Make charts and visualizations
import matplotlib.pyplot as plt

# Ensemble models
from sklearn.ensemble import (
    RandomForestClassifier, 
    AdaBoostClassifier,       
    GradientBoostingClassifier, 
    BaggingClassifier,     
    VotingClassifier,     
)

# Classic tree-based classifier
from sklearn.tree import DecisionTreeClassifier

# Support Vector Machine classifier
from sklearn.svm import SVC

# Logistic Regression classifier (good baseline linear model)
from sklearn.linear_model import LogisticRegression

# K-Nearest Neighbors classifier (based on closest points)
from sklearn.neighbors import KNeighborsClassifier

# Neural network classifier (Multi-Layer Perceptron)
from sklearn.neural_network import MLPClassifier

# Convert text labels into numeric form
from sklearn.preprocessing import LabelEncoder

# Split data into training and testing sets
from sklearn.model_selection import train_test_split

# Metrics to evaluate model performance
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [24]:
# Load the dataset ("../" up to notebook folder and then "..\" up to root repo folder)
df = pd.read_csv("../../data/winequality-red.csv", sep=";")

# Display structure and first few rows
df.info()
df.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5


### **Reflection 1: Data Inspection Summary**
- The dataset contains **1,599 rows** and **12 columns**, all representing chemical properties of red wine plus a quality score.
- There are **no missing values** in any column therefore no data cleaning is required at this stage.
- Most variables are stored as **float64**, while the target variable `quality` is an **integer**, which is appropriate for classification tasks.
- Features represent measurable chemical attributes such as acidity levels, sulfur dioxide content, density, and alcohol percentage.
- The first few rows show normal-looking numeric ranges with no obvious outliers or formatting issues.

### **Understanding the Target Variable**
- The target variable is **quality**, an integer score from 0 to 10 assigned by wine tasters.
- For modeling, we will convert this numeric score into **three classes**:
  - **Low quality:** 3–4
  - **Medium quality:** 5–6
  - **High quality:** 7–8
- This simplifies the problem into a clearer **classification task**.
- The dataset includes **1,599 samples** and **12 total columns** (11 features + 1 target).

## Section 2. Prepare the Data

In [25]:
# Define helper function that:
# Takes one input, the quality (which we will temporarily name q while in the function)
# And returns a string of the quality label (low, medium, high)
# This function will be used to create the quality_label column
def quality_to_label(q):
    if q <= 4:
        return "low"
    elif q <= 6:
        return "medium"
    else:
        return "high"
    
# Call the apply() method on the quality column to create the new quality_label column
df["quality_label"] = df["quality"].apply(quality_to_label)

# Then, create a numeric column for modeling: 0 = low, 1 = medium, 2 = high
def quality_to_number(q):
    if q <= 4:
        return 0
    elif q <= 6:
        return 1
    else:
        return 2
    
df["quality_numeric"] = df["quality"].apply(quality_to_number)

### **Reason for This Prep**
We convert the original wine quality scores into three clear groups—low, medium, and high—so the models have an easier classification task. Creating both a text label and a numeric version gives us flexibility: the label is easier to understand, and the numeric values are needed for training machine learning models.

- **Low quality**: 3–4  
- **Medium quality**: 5–6  
- **High quality**: 7–8  

- **`quality_label`** — a text version of the class (low, medium, high), which is easier to read in analysis.
- **`quality_numeric`** — a numeric encoding (0, 1, 2) required for training machine learning models.

This preprocessing step makes the dataset easier to work with and ensures that our models can learn meaningful patterns from the wine’s chemical features.

## Section 3. Feature Selection and Justification

In [26]:
# Define input features (X) and target (y)
# Features: all columns except 'quality' and 'quality_label' and 'quality_numberic' - drop these from the input array
# Target: quality_label (the new column we just created)
X = df.drop(columns=["quality", "quality_label", "quality_numeric"])  # Features
y = df["quality_numeric"]  # Target

### **Why these features?**
The wine dataset includes 11 chemical properties such as acidity, sugar, sulfur dioxide, density, pH, sulphates, and alcohol. These describe the measurable characteristics of each wine sample, so they make sense as predictors. 

I removed the following columns from the feature set:

- `quality` – original 0–10 score  
- `quality_label` – text labels (“low”, “medium”, “high”)  
- `quality_numeric` – encoded target we will predict  

These columns represent the actual quality rating, so they should not be included in the input features.

### **Target variable**
I used `quality_numeric` (0 = low, 1 = medium, 2 = high) as the target because most ML models require numeric labels for classification.

## Section 4. Split the Data into Train and Test

In [27]:
# Train/test split (stratify to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Section 5.  Evaluate Model Performance

## Section 6. Compare Results 

## Section 7. Conclusions and Insights